### Importing Libraries

In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import transforms
from torchvision.models import resnet18, ResNet18_Weights 
import opendatasets as od
from torch.optim import Adam

### Downloading Datasets

In [2]:
# od.download(dataset_id_or_url = 'https://www.kaggle.com/datasets/briscdataset/brisc2025', data_dir = f'{os.getcwd()}\Data')

### Creating data manifest and splitting data

#### 1. Train

In [3]:
%cd ..

d:\Projects\BrainTumor-Classification-Segmentation


In [4]:
train_image = []
train_label = []

root_dir = f'{os.getcwd()}/Data/classification_task/train'

for label in os.listdir(root_dir):
    class_dir = f'{root_dir}/{label}'
    for image in os.listdir(class_dir):
        train_label.append(label)
        train_image.append(f'{class_dir}/{image}')

In [5]:
train = pd.DataFrame(zip(train_image, train_label), columns = ['image', 'label'])

#### 2. Test

In [6]:
test_image = []
test_label = []

root_dir = f'{os.getcwd()}/Data/classification_task/test'

for label in os.listdir(root_dir):
    class_dir = f'{root_dir}/{label}'
    for image in os.listdir(class_dir):
        test_label.append(label)
        test_image.append(f'{class_dir}/{image}')

In [7]:
test = pd.DataFrame(zip(test_image, test_label), columns = ['image', 'label'])

#### 3. Validation

In [8]:
train, val = train_test_split(train, test_size = 0.3, random_state = 42, stratify = train['label'])
train = train.reset_index(drop = True)
val = val.reset_index(drop = True)

#### Data Showcase

In [9]:
print(f'Train: \n{train.head()}')
print(f'Test: \n{test.head()}')
print(f'Validation: \n{val.head()}')

Train: 
                                               image     label
0  d:\Projects\BrainTumor-Classification-Segmenta...    glioma
1  d:\Projects\BrainTumor-Classification-Segmenta...  no_tumor
2  d:\Projects\BrainTumor-Classification-Segmenta...  no_tumor
3  d:\Projects\BrainTumor-Classification-Segmenta...  no_tumor
4  d:\Projects\BrainTumor-Classification-Segmenta...    glioma
Test: 
                                               image   label
0  d:\Projects\BrainTumor-Classification-Segmenta...  glioma
1  d:\Projects\BrainTumor-Classification-Segmenta...  glioma
2  d:\Projects\BrainTumor-Classification-Segmenta...  glioma
3  d:\Projects\BrainTumor-Classification-Segmenta...  glioma
4  d:\Projects\BrainTumor-Classification-Segmenta...  glioma
Validation: 
                                               image       label
0  d:\Projects\BrainTumor-Classification-Segmenta...   pituitary
1  d:\Projects\BrainTumor-Classification-Segmenta...    no_tumor
2  d:\Projects\BrainTumor-Classif

### Dataset and Dataloader

In [10]:
train_transform = transforms.Compose([
    transforms.RandomRotation(degrees = 10),
    transforms.Resize((225, 225)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

val_transform = transforms.Compose([
    transforms.Resize((225, 225)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

le = LabelEncoder()
le.fit(train['label'])

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[object](4,)","['glioma','meningioma','no_tumor','pituitary']"


In [11]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        label = row['label']
        label = le.transform([label])[0]
        label = torch.tensor(label, dtype = torch.long)

        image = Image.open(row['image']).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [12]:
train = CustomDataset(train, train_transform)
test = CustomDataset(test, val_transform)
val = CustomDataset(val, val_transform)

In [13]:
train_loader = DataLoader(train, batch_size = 64, shuffle = True, pin_memory = True)
test_loader = DataLoader(test, batch_size = 64, shuffle = False, pin_memory = True)
val_loader = DataLoader(val, batch_size = 64, shuffle = False, pin_memory = True)

### Model initialization

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [15]:
model = resnet18(weights = ResNet18_Weights.DEFAULT)

In [16]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [17]:
for param in model.parameters():
    param.requires_grad = False

for name, param in model.named_parameters():
    if 'layer4' in name:
        param.requires_grad = True

model.fc = nn.Sequential(
    nn.Dropout(p = 0.2),
    nn.Linear(in_features = model.fc.in_features, out_features = 4)
)

model = model.to(device)

In [18]:
optimizer = Adam([
    {'params': [p for n, p in model.named_parameters() if 'fc' in n], 'lr': 1e-4},
    {'params': [p for n, p in model.named_parameters() if 'layer4' in n], 'lr': 1e-5}
])

criterian = nn.CrossEntropyLoss()

### Training

In [19]:
best_val_score = 0.0
patiance = 5
counter = 0

for epoch in range(30):
    model.train()
    total_loss = 0.0
    
    for image, label in train_loader:
        image, label = image.to(device), label.to(device)

        optimizer.zero_grad()
        output = model(image)

        loss = criterian(output, label)
        loss.backward()

        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for image, label in val_loader:
            image, label = image.to(device), label.to(device)
            output = model(image)

            pred = output.argmax(dim = 1)
            correct += (pred == label).sum().item()
            total += label.size(0)

    val_acc = correct / total
    print(f'Epoch: {epoch + 1}| Loss: {(total_loss / len(train_loader)):.4f}| Validation accuracy: {val_acc:.4f}')

    if val_acc > best_val_score:
        best_val_score = val_acc
        counter = 0
        print(f'Best Validation accuracy: {best_val_score:.4f}| Model saved at: artifacts/objects/')
        torch.save(model.state_dict(), 'artifacts/objects/model.pth')

    else:
        counter += 1
        if counter >= patiance:
            print(f'No improvement for {patiance}, breaking the training loop')
            break

Epoch: 1| Loss: 1.0069| Validation accuracy: 0.8033
Best Validation accuracy: 0.8033| Model saved at: artifacts/objects/
Epoch: 2| Loss: 0.5250| Validation accuracy: 0.8693
Best Validation accuracy: 0.8693| Model saved at: artifacts/objects/
Epoch: 3| Loss: 0.3848| Validation accuracy: 0.8993
Best Validation accuracy: 0.8993| Model saved at: artifacts/objects/
Epoch: 4| Loss: 0.3064| Validation accuracy: 0.9140
Best Validation accuracy: 0.9140| Model saved at: artifacts/objects/
Epoch: 5| Loss: 0.2509| Validation accuracy: 0.9233
Best Validation accuracy: 0.9233| Model saved at: artifacts/objects/
Epoch: 6| Loss: 0.2155| Validation accuracy: 0.9333
Best Validation accuracy: 0.9333| Model saved at: artifacts/objects/
Epoch: 7| Loss: 0.1835| Validation accuracy: 0.9380
Best Validation accuracy: 0.9380| Model saved at: artifacts/objects/
Epoch: 8| Loss: 0.1539| Validation accuracy: 0.9440
Best Validation accuracy: 0.9440| Model saved at: artifacts/objects/
Epoch: 9| Loss: 0.1364| Validati

### Evaluation

In [20]:
model.load_state_dict(torch.load('artifacts/objects/model.pth', map_location = device))
model = model.to(device)
model.eval()

test_pred, test_label = [], []
with torch.no_grad():
    for image, label in test_loader:
        image, label = image.to(device), label.to(device)

        output = model(image)
        pred = output.argmax(dim = 1)

        test_pred.extend(pred.cpu().numpy())
        test_label.extend(label.cpu().numpy())

test_acc = sum(p == l for p, l in zip(test_pred, test_label)) / len(test_label)
print(f'Test accuracy: {test_acc:.4f}')

print(f'Confusion matrix: \n{confusion_matrix(test_label, test_pred)}')
print(f'Classication report: \n{classification_report(test_label, test_pred, target_names = le.classes_)}')


Test accuracy: 0.9540
Confusion matrix: 
[[234  19   0   1]
 [  7 283   9   7]
 [  0   0 140   0]
 [  0   3   0 297]]
Classication report: 
              precision    recall  f1-score   support

      glioma       0.97      0.92      0.95       254
  meningioma       0.93      0.92      0.93       306
    no_tumor       0.94      1.00      0.97       140
   pituitary       0.97      0.99      0.98       300

    accuracy                           0.95      1000
   macro avg       0.95      0.96      0.96      1000
weighted avg       0.95      0.95      0.95      1000

